# NetraGraph: UNSW-NB15 GPU Training & Research Pipeline
This Google Colab notebook provides an end-to-end, GPU-accelerated training pipeline for the **UNSW-NB15** network intrusion dataset.

### Pipeline Steps:
1. Environment Setup & Requirements
2. GPU & CUDA Verification
3. Dataset Path Configuration & ZIP Audit
4. Data Integrity & Leakage Audit
5. Feature Preparation & Preprocessor Fitting
6. CatBoost GPU Classifier Training
7. Cybersecurity Metrics & False Positive Analysis
8. Confusion Matrix & Feature Importance Visualization
9. Model B Cross-Dataset Transferability Evaluation
10. NetraGraph-Compliant Versioned Artifact Bundling

## 1. Environment Setup & Dependency Installation

In [ ]:
# Clone repository if running standalone in Colab
import os, sys
from pathlib import Path

if not os.path.exists("/content/NetraGraph") and os.path.exists("/content"):
    print("Cloning NetraGraph repository...")
    !git clone https://github.com/Cyberdude441/NetraGraph.git /content/NetraGraph
    %cd /content/NetraGraph
elif os.path.exists("/content/NetraGraph"):
    %cd /content/NetraGraph

# Install GPU & Pipeline Requirements
!pip install -q -r training/unsw_nb15/requirements.txt

## 2. GPU & Hardware Diagnostics

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd() / "training" / "unsw_nb15"))
sys.path.insert(0, str(Path.cwd() / "backend"))

from utils import detect_hardware, print_hardware_status
hw = detect_hardware(requested_device="auto")
print_hardware_status(hw)

## 3. Dataset Configuration & Safe ZIP Inspection
Set `DATASET_DIR` to the directory containing your UNSW-NB15 CSV files or ZIP archive.

In [ ]:
# Configure path to user-supplied dataset
DATASET_DIR = Path("/content/UNSW-NB15")

# Fallback to local sample or current working directory if /content does not exist
if not DATASET_DIR.exists():
    DATASET_DIR = Path("./data/UNSW-NB15")
    DATASET_DIR.mkdir(parents=True, exist_ok=True)

print(f"Target Dataset Directory: {DATASET_DIR.resolve()}")

## 4. Data Loading, Schema Validation & Leakage Shield Audit

In [ ]:
from prepare_data import load_dataset_frames, audit_and_validate_data, prepare_features
from config import TrainingConfig

config = TrainingConfig(
    data_dir=str(DATASET_DIR),
    output_dir="artifacts/network-anomaly-unsw/v1",
    device=hw["training_device"],
    iterations=1000,
    depth=6,
    learning_rate=0.05,
)

train_df, test_df = load_dataset_frames(config.data_dir, subsample_ratio=config.subsample_ratio)
validation_report = audit_and_validate_data(train_df, test_df)

## 5. Feature Engineering & Preprocessor Preparation

In [ ]:
X_train, y_train, X_test, y_test, preprocessor, feature_names, cat_indices = prepare_features(
    train_df, test_df, config
)
print(f"Feature matrix prepared: {X_train.shape[1]} features")
print(f"Training samples: {X_train.shape[0]:,}, Testing samples: {X_test.shape[0] if X_test is not None else 0:,}")

## 6. Model Training (CatBoost GPU / CPU)

In [ ]:
from train import train_unsw_model

artifact_path = train_unsw_model(config)
print(f"Training complete! Artifact package saved at: {artifact_path}")

## 7. Forensic Cybersecurity Metric Evaluation

In [ ]:
from evaluate import evaluate_saved_artifact

metrics = evaluate_saved_artifact(artifact_path)

## 8. Cross-Dataset Validation (Existing Model B vs UNSW-NB15)

In [ ]:
from cross_validate_model_b import cross_validate_model_b_on_unsw

# Evaluate existing NSL-KDD Model B on UNSW-NB15
try:
    if test_df is not None:
        test_csv_sample = DATASET_DIR / "UNSW_NB15_testing-set.csv"
        if test_csv_sample.exists():
            cv_results = cross_validate_model_b_on_unsw(test_csv_sample, subsample_limit=5000)
except Exception as e:
    print(f"Cross-validation note: {e}")

## 9. Verification & Cryptographic Provenance Hash

In [ ]:
import json
from utils import calculate_sha256

sha_hash = calculate_sha256(artifact_path / "model.joblib")
metadata = json.loads((artifact_path / "metadata.json").read_text())

print("==================================================")
print("UNSW-NB15 ARTIFACT VERIFICATION SUMMARY")
print("==================================================")
print(f"Model Name     : {metadata.get('model_name')}")
print(f"Version        : {metadata.get('model_version')}")
print(f"Device Used    : {metadata.get('training_device')}")
print(f"SHA-256 Hash   : {sha_hash}")
print(f"Accuracy       : {metadata.get('training_metrics', {}).get('accuracy', 0)*100:.2f}%")
print(f"False Pos Rate : {metadata.get('training_metrics', {}).get('false_positive_rate', 0)*100:.3f}%")
print(f"False Neg Rate : {metadata.get('training_metrics', {}).get('false_negative_rate', 0)*100:.3f}%")
print("==================================================")